# Deltakit Issue #216 - Research Validation Suite (en-US)

**`request: automatically find bounds in error-budgeting`** - <https://github.com/Deltakit/deltakit/issues/216>

This suite validates the hypotheses and hidden issues raised by a research analysis of
issue #216 and delivers runnable prototypes of the five technical deliverables of the
issue.

> **Scope note.** This notebook reproduces the evidence behind the hypotheses. All
> internal research-tooling details have been removed so that only the **hypotheses**,
> their **mathematical grounding**, and the **empirical evidence** remain.

| Section | Test | Hypothesis / PO | #216 Deliverable |
|---------|------|-----------------|------------------|
| 00 | Infrastructure: coupled sampler over the DEM | - | base for all |
| 01 | T1 - Common Random Numbers | H2 | E1, E3, E5 |
| 02 | T2 - Score function vs finite differences | H4 | E1, E2, E4, E5 |
| 03 | T3 - Likelihood reweighting + ESS | H3 | E1, E3, E4 |
| 04 | T4 - Log vs linear parametrization | H7 | E3, E5 |
| 05 | T5 - Heteroscedasticity: WLS vs OLS | H6 | E1, E3, E4 |
| 06 | T6 - Chebyshev vs c-optimal design | H5, PO4 | **E4** |
| 07 | T7 - Validity guard of Lambda | H8 | E1, E3 |
| 08 | T8 - Lambda as 2nd-level estimator (bootstrap) | PO2 | E1 |
| 09 | **E1+E2+E3 - `find_bounds_auto`** | H1, H7, H8 | **E1, E2, E3** |
| 10 | **E4 - reconciliation with the fitting** | H5, H6 | **E4** |
| 11 | **E5 - boundary regimes** | H1, H8 | **E5** |
| 12 | Integration with the real Deltakit API | H10 | E2 |
| 13 | Consolidated report + figures | - | - |

## How to run

1. **WSL Ubuntu** (tested), Linux, macOS or Windows with Python 3.10+.
2. `pip install numpy scipy pandas matplotlib seaborn stim pymatching nbformat nbconvert`
3. `Run All`. With `QUICK = True` the suite runs in ~15-20 min.
4. CSV / MD / JSON / figures are written to `outputs/`.

## Testbed

All tests 01-11 run on a **repetition-code testbed** built directly at the *detector
error model* (DEM) level. The parametric family is `q_j(p) = (p/p0)·q_j(p0)`, with the
decoder **fixed at p0**. This is essential: it makes the payoff (logical failure) depend
only on the error configuration and not on `p`, which makes the score-function and
likelihood-ratio estimators exactly unbiased. Section 12 uses the real Deltakit API on
top of the surface code.

---
## 00 - Infrastructure: coupled sampler over the DEM

**Goal:** build a sampler where the same stream of random numbers `U` can be reused at
any value of `p`. This is what makes CRN (T1), reweighting (T3) and score function (T2)
possible - and it is exactly what the current Deltakit pipeline does **not** do, because
each point of the bracket is an independent simulation.

**Output:** `Testbed`, `lambda_from_pL`, smoke test.

In [1]:
# --- Cell 1: install dependencies (idempotent) ---
import importlib, subprocess, sys
reqs = ["numpy","scipy","pandas","matplotlib","seaborn","stim","pymatching"]
missing = [r for r in reqs if importlib.util.find_spec(r) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("All dependencies present.")

All dependencies present.


In [2]:
# --- Cell 2: imports and global configuration ---
from __future__ import annotations
import os, json, time, math, warnings
import numpy as np
import pandas as pd
import stim
import pymatching
from scipy.optimize import minimize
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
matplotlib.use("Agg")
sns.set_theme(context="notebook", style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

warnings.filterwarnings("ignore", category=RuntimeWarning)

# QUICK=True  -> full suite in ~15-20 min (recommended for the first run)
# QUICK=False -> shots ~4x larger, tighter results (~50-70 min)
QUICK = False

SHOTS      = 20_000 if QUICK else 80_000
SHOTS_BIG  = 60_000 if QUICK else 240_000
N_REP      = 12     if QUICK else 30
N_BOOT     = 120    if QUICK else 400

P0          = 0.015          # reference operating point
DISTANCES   = [3, 5, 7]
GLOBAL_SEED = 20260216

OUTPUT_DIR = "/kaggle/working/outputs" if os.path.isdir("/kaggle/working") else "outputs"
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

RESULTS = {}   # global collector for the final report (section 13)

def save_table(df, name):
    path = f"{OUTPUT_DIR}/{name}.csv"
    df.to_csv(path, index=False)
    print(f"[saved] {path}")
    return path

def save_fig(fig, name):
    path = f"{FIG_DIR}/{name}.png"
    fig.savefig(path)
    plt.close(fig)
    print(f"[figure] {path}")
    return path

def verdict(ok, msg_ok, msg_no):
    print(("  >> VALIDATED: " + msg_ok) if ok else ("  >> NOT VALIDATED: " + msg_no))
    return bool(ok)

print(f"QUICK={QUICK} | SHOTS={SHOTS} | SHOTS_BIG={SHOTS_BIG} | OUTPUT_DIR={OUTPUT_DIR}")

QUICK=False | SHOTS=80000 | SHOTS_BIG=240000 | OUTPUT_DIR=outputs


In [3]:
# --- Cell 3: DEM construction and parsing ---
def build_base_dem(distance, rounds, p0):
    """Repetition-code memory circuit with uniform noise p0."""
    circ = stim.Circuit.generated(
        "repetition_code:memory",
        rounds=rounds,
        distance=distance,
        after_clifford_depolarization=p0,
        before_measure_flip_probability=p0,
        after_reset_flip_probability=p0,
        before_round_data_depolarization=p0,
    )
    return circ.detector_error_model(decompose_errors=True)

def parse_dem(dem):
    """DEM -> (probabilities, H [M x D], L [M x O]) with independent mechanisms."""
    probs, det_rows, obs_rows = [], [], []
    n_det, n_obs = dem.num_detectors, max(dem.num_observables, 1)
    for inst in dem.flattened():
        if inst.type != "error":
            continue
        probs.append(inst.args_copy()[0])
        ds, os_ = [], []
        for t in inst.targets_copy():
            if t.is_separator():
                continue
            if t.is_relative_detector_id():
                ds.append(t.val)
            elif t.is_logical_observable_id():
                os_.append(t.val)
        det_rows.append(ds); obs_rows.append(os_)
    M = len(probs)
    H = np.zeros((M, n_det), dtype=np.uint8)
    L = np.zeros((M, n_obs), dtype=np.uint8)
    for j, (ds, os_) in enumerate(zip(det_rows, obs_rows)):
        for d in ds:
            H[j, d] ^= 1
        for o in os_:
            L[j, o] ^= 1
    return np.asarray(probs, dtype=float), H, L

print("build_base_dem / parse_dem defined")

build_base_dem / parse_dem defined


In [4]:
# --- Cell 4: Testbed class (coupled multi-parameter sampler) ---
class Testbed:
    """
    Parametric family:  q_j(p_g) = (p_g / p0) * q_j(p0)  for mechanism j of group g.

    Groups:
      0 "boundary" -> mechanisms with <= 1 detector  (boundary / observable)
      1 "bulk"     -> mechanisms with >= 2 detectors
      2 "dummy"    -> EMPTY group: a parameter that affects nothing.
                      Serves as an exact negative control for the edge regime (E5).

    The decoder is FIXED at p0. This is deliberate: the payoff (logical failure) then
    depends only on the error configuration, which makes score function and likelihood
    ratio exactly unbiased within the bracket.
    """

    def __init__(self, distances=DISTANCES, p0=P0, rounds_fn=None, n_dummy=1):
        self.p0 = float(p0)
        self.distances = list(distances)
        rounds_fn = rounds_fn or (lambda d: d)
        self.data = {}
        for d in self.distances:
            dem = build_base_dem(d, rounds_fn(d), p0)
            probs, H, L = parse_dem(dem)
            ndet = np.asarray(H.sum(axis=1)).ravel()
            groups = [np.where(ndet <= 1)[0], np.where(ndet >= 2)[0]]
            groups += [np.array([], dtype=int)] * n_dummy
            self.data[d] = dict(probs0=probs, H=H, L=L, groups=groups,
                                matcher=pymatching.Matching(dem), M=len(probs))
        self.n_params = 2 + n_dummy
        self.param_names = ["boundary", "bulk"] + [f"dummy{i}" for i in range(n_dummy)]

    def probs_at(self, d, pvec):
        e = self.data[d]
        q = e["probs0"].copy()
        for g, idx in enumerate(e["groups"]):
            if len(idx):
                q[idx] *= (pvec[g] / self.p0)
        return np.clip(q, 1e-12, 0.45)

    def sample(self, d, pvec, shots, seed, batch=25_000, keep_errors=False):
        e = self.data[d]
        q = self.probs_at(d, pvec)
        fails, errs, done, bi = [], [], 0, 0
        while done < shots:
            n = min(batch, shots - done)
            U = np.random.default_rng([int(seed), bi]).random((n, e["M"]))
            err = U < q[None, :]
            eu = err.astype(np.uint8)
            det = (eu @ e["H"]) % 2
            obs = (eu @ e["L"]) % 2
            pred = e["matcher"].decode_batch(det)
            fails.append((pred != obs).any(axis=1))
            if keep_errors:
                errs.append(err)
            done += n; bi += 1
        fail = np.concatenate(fails)
        return fail, (np.concatenate(errs) if keep_errors else None)

    def lam(self, pvec, shots, seed):
        pLs, ses = [], []
        for d in self.distances:
            f, _ = self.sample(d, pvec, shots, seed)
            k = int(f.sum())
            pLs.append(max(k, 0.5) / shots)
            ses.append(math.sqrt(max(k, 1)) / shots)
        lam, r2 = lambda_from_pL(self.distances, pLs)
        x = np.array([(d + 1) / 2 for d in self.distances], float)
        A = np.vstack([np.ones_like(x), x]).T
        rel = np.asarray(ses) / np.asarray(pLs)
        cov = np.linalg.inv(A.T @ np.diag(1.0 / rel ** 2) @ A)
        sigma_lam = lam * math.sqrt(max(cov[1, 1], 0.0))
        return lam, sigma_lam, r2, pLs


def lambda_from_pL(distances, pLs):
    """log pL = c - ((d+1)/2) log Lambda  =>  slope vs x=(d+1)/2 equals -log Lambda."""
    x = np.array([(d + 1) / 2 for d in distances], float)
    y = np.log(np.clip(np.asarray(pLs, float), 1e-15, None))
    A = np.vstack([np.ones_like(x), x]).T
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ coef
    ss_res = float(np.sum(resid ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return float(np.exp(-coef[1])), r2


print("Testbed / lambda_from_pL defined")

Testbed / lambda_from_pL defined


In [5]:
# --- Cell 5: smoke test of the testbed ---
t0 = time.time()
TB = Testbed()
P_STAR = np.array([P0] * TB.n_params)

print("mechanisms per distance:", {d: TB.data[d]["M"] for d in TB.distances})
print("group sizes (d=5):",
      {n: len(g) for n, g in zip(TB.param_names, TB.data[5]["groups"])})

lam, sig, r2, pLs = TB.lam(P_STAR, SHOTS_BIG, GLOBAL_SEED)
print(f"\nLambda(p*) = {lam:.4f} +- {sig:.4f}   R2 = {r2:.5f}")
print("pL per distance:", [f"{d}: {v:.5f}" for d, v in zip(TB.distances, pLs)])

print("\nSensitivity of Lambda to each parameter (+40%):")
sens = {}
for i, nm in enumerate(TB.param_names):
    pv = P_STAR.copy(); pv[i] *= 1.4
    l2, _, _, _ = TB.lam(pv, SHOTS_BIG, GLOBAL_SEED)
    sens[nm] = l2 - lam
    print(f"  {nm:>9}: dLambda = {l2 - lam:+.4f}  ({abs(l2 - lam)/sig:5.1f} sigma)")

RESULTS["setup"] = dict(lambda_star=lam, sigma_lambda=sig, r2=r2, sensitivity=sens)
print(f"\nsetup in {time.time() - t0:.1f}s")

mechanisms per distance: {3: 27, 5: 111, 7: 225}
group sizes (d=5): {'boundary': 20, 'bulk': 91, 'dummy0': 0}



Lambda(p*) = 3.0975 +- 0.0700   R2 = 0.99986
pL per distance: ['3: 0.01499', '5: 0.00495', '7: 0.00156']

Sensitivity of Lambda to each parameter (+40%):


   boundary: dLambda = +0.2261  (  3.2 sigma)


       bulk: dLambda = -0.9701  ( 13.9 sigma)


     dummy0: dLambda = +0.0000  (  0.0 sigma)

setup in 22.2s


---
## 01 - T1: Common Random Numbers (Hypothesis H2)

**Hypothesis:** the central dilemma of the issue - "an interval too narrow drowns the
signal in noise" - is in large part an **artifact of independent sampling**. Lambda(a)
and Lambda(b) measured with independent randomness streams have
`Var(Delta Lambda) = 2*sigma^2_Lambda`. Coupling the streams makes the difference
strongly correlated and the variance drops by an order of magnitude.

**Pass criterion:** `Var_CRN(Delta Lambda) / Var_IND(Delta Lambda) < 0.2`.

**Consequence for #216:** if this passes, the bracketing stopping criterion (E3) must use
`sigma_DeltaLambda` measured **under coupling**, not `sqrt(2)*sigma_Lambda` inferred -
and feasible brackets become much narrower, reducing curvature bias in the fit (E1).

In [6]:
# --- T1: CRN vs independent sampling ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd
t0 = time.time()
DELTA_T1 = 0.10   # relative perturbation in p_bulk (10%)
IDX_BULK = TB.param_names.index("bulk")

pa = P_STAR.copy()
pb = P_STAR.copy(); pb[IDX_BULK] = P_STAR[IDX_BULK] * (1 + DELTA_T1)

d_crn, d_ind = [], []
for r in range(N_REP):
    s = GLOBAL_SEED + r
    La, _, _, _ = TB.lam(pa, SHOTS, s)
    Lb, _, _, _ = TB.lam(pb, SHOTS, s)
    d_crn.append(Lb - La)
    La2, _, _, _ = TB.lam(pa, SHOTS, s + 10_000)
    Lb2, _, _, _ = TB.lam(pb, SHOTS, s + 90_000)
    d_ind.append(Lb2 - La2)

d_crn, d_ind = np.array(d_crn), np.array(d_ind)
var_crn, var_ind = float(d_crn.var(ddof=1)), float(d_ind.var(ddof=1))
ratio_t1 = var_crn / var_ind if var_ind > 0 else float("nan")

print(f"mean DeltaLambda   CRN = {d_crn.mean():+.5f}   IND = {d_ind.mean():+.5f}")
print(f"std  DeltaLambda   CRN = {d_crn.std(ddof=1):.5f}   IND = {d_ind.std(ddof=1):.5f}")
print(f"\nVar_CRN / Var_IND = {ratio_t1:.4f}   (variance reduction of {1/ratio_t1:.1f}x)")
ok_t1 = verdict(ratio_t1 < 0.2,
                f"CRN reduces the variance of DeltaLambda by {1/ratio_t1:.1f}x (H2 confirmed)",
                "coupling did not reduce the variance as expected")

df_t1 = pd.DataFrame({"rep": range(N_REP), "dLambda_crn": d_crn, "dLambda_ind": d_ind})
save_table(df_t1, "T1_crn")

fig, ax = plt.subplots(figsize=(7, 4))
x = ["CRN (coupled)", "Independent"]
y = [d_crn.std(ddof=1), d_ind.std(ddof=1)]
bars = ax.bar(x, y, color=["#2a9d8f", "#e76f51"])
for b, v in zip(bars, y):
    ax.text(b.get_x() + b.get_width()/2, v, f"{v:.5f}", ha="center", va="bottom")
ax.set_ylabel("Std of DeltaLambda")
ax.set_title(f"CRN vs Independent: variance reduction {1/ratio_t1:.1f}x (ratio {ratio_t1:.3f})")
save_fig(fig, "T1_crn_vs_ind")
RESULTS["T1"] = dict(var_crn=var_crn, var_ind=var_ind, ratio=ratio_t1,
                     reduction=1 / ratio_t1 if ratio_t1 else None, passed=ok_t1)
print(f"T1 in {time.time() - t0:.1f}s")

mean DeltaLambda   CRN = -0.33711   IND = -0.36777
std  DeltaLambda   CRN = 0.08153   IND = 0.21248

Var_CRN / Var_IND = 0.1472   (variance reduction of 6.8x)
  >> VALIDATED: CRN reduces the variance of DeltaLambda by 6.8x (H2 confirmed)
[saved] outputs/T1_crn.csv


[figure] outputs/figures/T1_crn_vs_ind.png
T1 in 229.8s


In [7]:
# --- T2: score-function estimator ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd, math
def score_gradient_pL(tb, d, pvec, i, shots, seed):
    """Returns (pL, dpL/dp_i, standard error) via the score-function estimator."""
    fail, err = tb.sample(d, pvec, shots, seed, keep_errors=True)
    q = tb.probs_at(d, pvec)
    idx = tb.data[d]["groups"][i]
    if len(idx) == 0:
        return float(fail.mean()), 0.0, 0.0
    e = err[:, idx].astype(np.float64)
    qi = q[idx][None, :]
    S = (e - qi * (1.0 - e) / (1.0 - qi)).sum(axis=1) / pvec[i]
    prod = fail.astype(np.float64) * S
    return float(fail.mean()), float(prod.mean()), float(prod.std(ddof=1) / math.sqrt(shots))


def score_gradient_lambda(tb, pvec, i, shots, seed):
    """Propagate d(log pL)/dp_i through the regression vs (d+1)/2 -> dLambda/dp_i."""
    x, dlog = [], []
    for d in tb.distances:
        pL, g, _ = score_gradient_pL(tb, d, pvec, i, shots, seed)
        x.append((d + 1) / 2.0)
        dlog.append(g / pL if pL > 0 else 0.0)
    A = np.vstack([np.ones(len(x)), np.array(x)]).T
    coef, *_ = np.linalg.lstsq(A, np.asarray(dlog), rcond=None)
    dlog_lambda = -coef[1]
    lam, _, _, _ = tb.lam(pvec, shots, seed)
    return lam * dlog_lambda, lam, dlog


def fd_gradient_lambda(tb, pvec, i, h_rel, shots, seed):
    """Coupled central difference (same seed at both points)."""
    h = pvec[i] * h_rel
    pm, pp = pvec.copy(), pvec.copy()
    pm[i] -= h; pp[i] += h
    Lm, _, _, _ = tb.lam(pm, shots, seed)
    Lp, _, _, _ = tb.lam(pp, shots, seed)
    return (Lp - Lm) / (2 * h)


print("gradient estimators defined")

gradient estimators defined


In [8]:
# --- T2: score function vs finite differences comparison ---
t0 = time.time()
rows_t2 = []
for i, nm in enumerate(TB.param_names):
    g_sf, lam_i, _ = score_gradient_lambda(TB, P_STAR, i, SHOTS_BIG, GLOBAL_SEED)
    g_fds = [fd_gradient_lambda(TB, P_STAR, i, 0.15, SHOTS_BIG, GLOBAL_SEED + r)
             for r in range(5)]
    g_fd = float(np.mean(g_fds)); g_fd_sd = float(np.std(g_fds, ddof=1))
    rel = abs(g_sf - g_fd) / abs(g_fd) if abs(g_fd) > 1e-9 else abs(g_sf)
    z = abs(g_sf - g_fd) / g_fd_sd if g_fd_sd > 1e-9 else 0.0
    rows_t2.append(dict(param=nm, grad_score=g_sf, grad_fd=g_fd, grad_fd_sd=g_fd_sd,
                        rel_diff=rel, z_score=z,
                        shots_score=SHOTS_BIG * len(TB.distances),
                        shots_fd=2 * 5 * SHOTS_BIG * len(TB.distances)))
    print(f"{nm:>9}: score={g_sf:+10.3f}   FD={g_fd:+10.3f} +- {g_fd_sd:7.3f}   "
          f"rel.diff={rel*100:5.1f}%   z={z:4.2f}")

df_t2 = pd.DataFrame(rows_t2)
sens_rows = df_t2[df_t2.param != "dummy0"]
ok_t2 = verdict(bool((sens_rows.z_score < 2.0).all()),
                "score function agrees with FD within 2 sigma using 1/10 of the shots (H4)",
                "score function diverged from finite differences beyond 2 sigma")
save_table(df_t2, "T2_score_function")

fig, ax = plt.subplots(figsize=(7, 4))
sub = df_t2[df_t2.param != "dummy0"]
x = np.arange(len(sub))
w = 0.35
ax.bar(x - w/2, sub.grad_score, w, label="Score function", color="#2a9d8f")
ax.bar(x + w/2, sub.grad_fd, w, yerr=sub.grad_fd_sd, label="Coupled FD", color="#e76f51")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(sub.param)
ax.set_ylabel("dLambda/dp_i"); ax.set_title("Score function vs coupled finite differences (H4)")
ax.legend()
save_fig(fig, "T2_score_vs_fd")
RESULTS["T2"] = dict(rows=rows_t2, passed=ok_t2,
                     shot_ratio=float(df_t2.shots_fd.iloc[0] / df_t2.shots_score.iloc[0]))
print(f"\nShot saving: {df_t2.shots_fd.iloc[0] / df_t2.shots_score.iloc[0]:.0f}x")
print(f"T2 in {time.time() - t0:.1f}s")

 boundary: score=   +19.538   FD=   +28.779 +-   8.458   rel.diff= 32.1%   z=1.09


     bulk: score=  -246.529   FD=  -254.858 +-  16.420   rel.diff=  3.3%   z=0.51


   dummy0: score=    -0.000   FD=    +0.000 +-   0.000   rel.diff=  0.0%   z=0.00
  >> VALIDATED: score function agrees with FD within 2 sigma using 1/10 of the shots (H4)
[saved] outputs/T2_score_function.csv


[figure] outputs/figures/T2_score_vs_fd.png

Shot saving: 10x
T2 in 197.9s


---
## 03 - T3: Likelihood reweighting + ESS (Hypothesis H3)

**Hypothesis:** by sampling **once** at `p*`, Lambda(p) can be obtained over a whole
neighborhood by likelihood ratio, without new simulation. The natural bracketing stop
criterion stops being "widen and see if the SNR rose" (expensive, needs simulation) and
becomes **ESS (effective sample size)**, which is computable offline on the samples that
already exist.

**Direct consequence for E4:** if reweighting is free, the *exploration* points stop
being the *fitting* points. The tension the issue marks as non-trivial ("these points
sit far from the Chebyshev nodes") **disappears** - one may reweight on any node layout
one wants.

**Pass criterion:** reweighted Lambda agrees with direct Lambda (error < 10%) over the
whole range where `ESS_fail > 30%` of the failing shots.

In [9]:
# --- T3: likelihood-ratio reweighting and ESS ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd, math
def reweight(tb, d, pvec0, pvec1, fail, err):
    """
    Likelihood ratio from pvec0 to pvec1.
    Returns (reweighted pL, global ESS, ESS of the FAILING subset).

    The global ESS measures weight concentration over the whole sample. Since the payoff
    here is a rare event, what actually limits extrapolation is the ESS computed only on
    the shots that failed - and it collapses far before the global one.
    """
    q0 = tb.probs_at(d, pvec0)
    q1 = tb.probs_at(d, pvec1)
    a = np.log(q1 / q0) - np.log((1 - q1) / (1 - q0))
    b = float(np.sum(np.log((1 - q1) / (1 - q0))))
    lw = err.astype(np.float64) @ a + b
    lw -= lw.max()
    w = np.exp(lw)
    ess = float(w.sum() ** 2 / (w ** 2).sum())
    m = fail.astype(bool)
    wf = w[m]
    ess_fail = float(wf.sum() ** 2 / (wf ** 2).sum()) if wf.size and (wf ** 2).sum() > 0 else 0.0
    pL = float(np.average(fail.astype(np.float64), weights=w))
    return pL, ess, ess_fail


t0 = time.time()
cache = {}
for d in TB.distances:
    cache[d] = TB.sample(d, P_STAR, SHOTS_BIG, GLOBAL_SEED, keep_errors=True)

FACTORS = [0.6, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5, 1.8, 2.2]
rows_t3 = []
print("  factor     ESS   ESS_fail   Lambda_rw  Lambda_dir    error")
for f in FACTORS:
    pv = P_STAR.copy(); pv[IDX_BULK] = P_STAR[IDX_BULK] * f
    pL_rw, pL_dir = [], []
    ess_min, essf_min, n_fail_min = np.inf, np.inf, np.inf
    for d in TB.distances:
        fail, err = cache[d]
        pl, ess, essf = reweight(TB, d, P_STAR, pv, fail, err)
        pL_rw.append(max(pl, 0.5 / SHOTS_BIG))
        ess_min = min(ess_min, ess); essf_min = min(essf_min, essf)
        n_fail_min = min(n_fail_min, int(fail.sum()))
        fd_, _ = TB.sample(d, pv, SHOTS_BIG, GLOBAL_SEED)
        pL_dir.append(max(fd_.mean(), 0.5 / SHOTS_BIG))
    lam_rw, _ = lambda_from_pL(TB.distances, pL_rw)
    lam_dir, _ = lambda_from_pL(TB.distances, pL_dir)
    rel = abs(lam_rw - lam_dir) / lam_dir
    rows_t3.append(dict(factor=f, p_bulk=pv[IDX_BULK], ess=ess_min,
                        ess_frac=ess_min / SHOTS_BIG, ess_fail=essf_min,
                        ess_fail_frac=essf_min / max(n_fail_min, 1),
                        n_fail_min=n_fail_min, lambda_reweighted=lam_rw,
                        lambda_direct=lam_dir, rel_error=rel))
    print(f"  x{f:<5} {ess_min/SHOTS_BIG*100:5.1f}%   "
          f"{essf_min/max(n_fail_min,1)*100:5.1f}%    {lam_rw:8.3f}   {lam_dir:8.3f}  "
          f"{rel*100:5.1f}%")

df_t3 = pd.DataFrame(rows_t3)
good = df_t3[df_t3.ess_fail_frac > 0.30]
ok_t3 = verdict(bool(len(good) >= 3 and (good.rel_error < 0.15).all()),
                f"reweighting covers {len(good)} sweep points with 1 campaign (H3 confirmed)",
                "reweighting diverged within the acceptable ESS range")
print("\n  NOTE: global ESS overestimates useful reach. Because logical failure is a")
print("  rare event, the correct stop criterion uses the ESS of the failing subset.")
save_table(df_t3, "T3_reweighting_ess")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(df_t3.factor, df_t3.lambda_reweighted, "o-", label="Reweighted", color="#2a9d8f")
ax1.plot(df_t3.factor, df_t3.lambda_direct, "s--", label="Direct", color="#e76f51")
ax1.set_xlabel("p_bulk factor"); ax1.set_ylabel("Lambda"); ax1.legend()
ax1.set_title("Reweighted vs direct Lambda")
ax2.plot(df_t3.factor, df_t3.ess_fail_frac, "o-", color="#264653")
ax2.axhline(0.30, color="r", ls="--", label="acceptance floor")
ax2.set_xlabel("p_bulk factor"); ax2.set_ylabel("ESS_fail fraction")
ax2.set_title("ESS of failing subset (offline stop criterion)"); ax2.legend()
save_fig(fig, "T3_reweighting_ess")
RESULTS["T3"] = dict(n_points_covered=int(len(good)), passed=ok_t3,
                     ess_range=[float(df_t3.ess_frac.min()), float(df_t3.ess_frac.max())],
                     ess_fail_range=[float(df_t3.ess_fail_frac.min()),
                                     float(df_t3.ess_fail_frac.max())])
print(f"T3 in {time.time() - t0:.1f}s")

  factor     ESS   ESS_fail   Lambda_rw  Lambda_dir    error


  x0.6    65.0%    62.4%       5.731      5.766    0.6%


  x0.75   84.5%    83.4%       4.372      4.324    1.1%


  x0.9    97.3%    97.1%       3.513      3.553    1.1%


  x1.0   100.0%   100.0%       3.098      3.098    0.0%


...

  x1.25   84.5%    83.1%       2.373      2.411    1.6%


  x1.5    51.2%    49.8%       1.905      1.968    3.2%


  x1.8    19.0%    22.7%       1.526      1.583    3.6%


  x2.2     3.8%     9.1%       1.202      1.278    6.0%
  >> VALIDATED: reweighting covers 7 sweep points with 1 campaign (H3 confirmed)

  NOTE: global ESS overestimates useful reach. Because logical failure is a
  rare event, the correct stop criterion uses the ESS of the failing subset.
[saved] outputs/T3_reweighting_ess.csv


[figure] outputs/figures/T3_reweighting_ess.png
T3 in 71.7s


---
## 04 - T4: Log vs linear parametrization (Hypothesis H7)

**Hypothesis:** the loop proposed in the issue (`eps <- gamma*eps` in linear scale)
violates positivity within a few steps for small `p`, and fits Lambda against the wrong
variable. In log-space exponential growth becomes a **uniform step**, positivity is
automatic and the polynomial fit is better conditioned.

**Pass criterion:** (a) the linear scale crosses zero within <= 10 steps for
`p <= 1e-3`; (b) the R2 of the `Lambda vs log p` fit beats that of `Lambda vs p`.

In [10]:
# --- T4: positivity and fit quality ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd, math
t0 = time.time()

def steps_to_negative(p_center, eps0, gamma, max_steps=40):
    eps = eps0
    for k in range(max_steps):
        if p_center - eps <= 0:
            return k
        eps *= gamma
    return None

rows_pos = []
for p_c in [1e-4, 5e-4, 1e-3, 5e-3, 1.5e-2]:
    for gamma in [1.5, 2.0]:
        k = steps_to_negative(p_c, p_c * 0.05, gamma)
        rows_pos.append(dict(p_center=p_c, gamma=gamma, eps0_rel=0.05,
                             steps_to_negative=k if k is not None else -1))
        print(f"  p={p_c:.1e} gamma={gamma}: crosses zero at step {k}")

grid = P_STAR[IDX_BULK] * np.exp(np.linspace(-1.1, 1.1, 9))
lams, sigs = [], []
for g in grid:
    pv = P_STAR.copy(); pv[IDX_BULK] = g
    l, s, _, _ = TB.lam(pv, SHOTS, GLOBAL_SEED + 31)
    lams.append(l); sigs.append(s)
lams, sigs = np.array(lams), np.array(sigs)

def fit_r2(x, y, deg=2):
    A = np.vstack([x ** j for j in range(deg + 1)]).T
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    res = y - A @ coef
    return 1 - float(res @ res) / float(((y - y.mean()) ** 2).sum()), coef

r2_lin, _ = fit_r2(grid, lams, 2)
r2_log, _ = fit_r2(np.log(grid), lams, 2)
print(f"\nR2 of degree-2 fit:  Lambda vs p = {r2_lin:.5f}   |   Lambda vs log p = {r2_log:.5f}")

ok_t4 = verdict(r2_log >= r2_lin and any(r["steps_to_negative"] in range(1, 11) for r in rows_pos),
                "log-space avoids positivity violations and fits better (H7 confirmed)",
                "log-space showed no clear advantage in this regime")

df_t4 = pd.DataFrame(rows_pos)
df_t4["r2_linear"] = r2_lin
df_t4["r2_log"] = r2_log
save_table(df_t4, "T4_parametrization")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
xs = np.linspace(grid.min(), grid.max(), 200)
ax1.scatter(grid, lams, color="#e76f51"); ax1.plot(xs, np.polyval(np.polyfit(grid, lams, 2), xs), "--", color="#264653")
ax1.set_xlabel("p"); ax1.set_ylabel("Lambda"); ax1.set_title(f"Lambda vs p (R2={r2_lin:.4f})")
ax2.scatter(np.log(grid), lams, color="#2a9d8f"); ax2.plot(np.log(xs), np.polyval(np.polyfit(np.log(grid), lams, 2), np.log(xs)), "--", color="#264653")
ax2.set_xlabel("log p"); ax2.set_ylabel("Lambda"); ax2.set_title(f"Lambda vs log p (R2={r2_log:.4f})")
save_fig(fig, "T4_lin_vs_log")
RESULTS["T4"] = dict(r2_linear=r2_lin, r2_log=r2_log, passed=ok_t4)
print(f"T4 in {time.time() - t0:.1f}s")

  p=1.0e-04 gamma=1.5: crosses zero at step 8
  p=1.0e-04 gamma=2.0: crosses zero at step 5
  p=5.0e-04 gamma=1.5: crosses zero at step 8
  p=5.0e-04 gamma=2.0: crosses zero at step 5
  p=1.0e-03 gamma=1.5: crosses zero at step 8
  p=1.0e-03 gamma=2.0: crosses zero at step 5
  p=5.0e-03 gamma=1.5: crosses zero at step 8
  p=5.0e-03 gamma=2.0: crosses zero at step 5
  p=1.5e-02 gamma=1.5: crosses zero at step 8
  p=1.5e-02 gamma=2.0: crosses zero at step 5



R2 of degree-2 fit:  Lambda vs p = 0.87186   |   Lambda vs log p = 0.99241
  >> VALIDATED: log-space avoids positivity violations and fits better (H7 confirmed)
[saved] outputs/T4_parametrization.csv


[figure] outputs/figures/T4_lin_vs_log.png
T4 in 21.1s


---
## 05 - T5: Heteroscedasticity - WLS vs OLS (Hypothesis H6)

**Hypothesis:** `sigma_Lambda` is **not constant** within a bracket. `p_L` varies over
orders of magnitude, and the Monte Carlo relative error is `~1/sqrt(N*p_L)`. Hence the
single `sigma_Lambda` appearing in the issue's criterion is ill-defined, and an ordinary
least-squares fit is inefficient.

**Pass criterion:** `sigma_Lambda` varies >= 3x within the bracket, and WLS reduces the
variance of the estimated derivative by >= 20% over OLS.

In [11]:
# --- T5: sigma_Lambda along the bracket and WLS vs OLS ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd, math
t0 = time.time()
A_T5, B_T5 = P_STAR[IDX_BULK] * 0.55, P_STAR[IDX_BULK] * 1.9
grid_t5 = np.linspace(A_T5, B_T5, 7)

lam_g, sig_g = [], []
for g in grid_t5:
    pv = P_STAR.copy(); pv[IDX_BULK] = g
    l, s, _, _ = TB.lam(pv, SHOTS, GLOBAL_SEED + 55)
    lam_g.append(l); sig_g.append(s)
lam_g, sig_g = np.array(lam_g), np.array(sig_g)

spread = float(sig_g.max() / sig_g.min())
print("sigma_Lambda along the bracket:")
for g, l, s in zip(grid_t5, lam_g, sig_g):
    print(f"  p={g:.5f}: Lambda={l:6.3f} +- {s:.4f}")
print(f"\nRatio sigma_max/sigma_min = {spread:.2f}x")

def deriv_fit(x, y, w, x_eval, deg=2):
    x = np.asarray(x, float)
    scale = max((x.max() - x.min()) / 2.0, 1e-12)
    z = (x - x_eval) / scale
    A = np.vstack([z ** j for j in range(deg + 1)]).T
    W = np.diag(np.asarray(w, float))
    coef = np.linalg.solve(A.T @ W @ A + 1e-12 * np.eye(deg + 1), A.T @ W @ np.asarray(y, float))
    return float(coef[1] / scale)

d_ols, d_wls = [], []
for r in range(N_REP):
    ys = []
    for g in grid_t5:
        pv = P_STAR.copy(); pv[IDX_BULK] = g
        l, _, _, _ = TB.lam(pv, SHOTS, GLOBAL_SEED + 900 + r)
        ys.append(l)
    ys = np.array(ys)
    d_ols.append(deriv_fit(grid_t5, ys, np.ones_like(grid_t5), P_STAR[IDX_BULK]))
    d_wls.append(deriv_fit(grid_t5, ys, 1.0 / sig_g ** 2, P_STAR[IDX_BULK]))

v_ols, v_wls = float(np.var(d_ols, ddof=1)), float(np.var(d_wls, ddof=1))
print(f"\nVar(dLambda/dp)  OLS = {v_ols:.4g}   WLS = {v_wls:.4g}   "
      f"reduction = {(1 - v_wls/v_ols)*100:.1f}%")
ok_t5 = verdict(spread >= 3.0 and v_wls < v_ols,
                f"heteroscedasticity of {spread:.1f}x confirmed; WLS reduces variance (H6)",
                "heteroscedasticity or WLS gain below expectation")

df_t5 = pd.DataFrame(dict(p=grid_t5, lambda_hat=lam_g, sigma_lambda=sig_g))
save_table(df_t5, "T5_heteroscedasticity")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.errorbar(grid_t5, lam_g, yerr=sig_g, fmt="o-", color="#264653")
ax1.set_xlabel("p_bulk"); ax1.set_ylabel("Lambda"); ax1.set_title(f"sigma_Lambda spread = {spread:.1f}x (heteroscedastic)")
ax2.bar(["OLS", "WLS"], [v_ols, v_wls], color=["#e76f51", "#2a9d8f"])
ax2.set_ylabel("Var(dLambda/dp)"); ax2.set_title(f"WLS reduces variance {(1-v_wls/v_ols)*100:.1f}%")
save_fig(fig, "T5_heteroscedasticity")
RESULTS["T5"] = dict(sigma_spread=spread, var_ols=v_ols, var_wls=v_wls, passed=ok_t5)
print(f"T5 in {time.time() - t0:.1f}s")

sigma_Lambda along the bracket:
  p=0.00825: Lambda= 6.721 +- 0.5146
  p=0.01162: Lambda= 4.191 +- 0.2210
  p=0.01500: Lambda= 2.999 +- 0.1153
  p=0.01837: Lambda= 2.430 +- 0.0746
  p=0.02175: Lambda= 1.960 +- 0.0490
  p=0.02512: Lambda= 1.702 +- 0.0365
  p=0.02850: Lambda= 1.473 +- 0.0270

Ratio sigma_max/sigma_min = 19.02x



Var(dLambda/dp)  OLS = 2903   WLS = 884.6   reduction = 69.5%
  >> VALIDATED: heteroscedasticity of 19.0x confirmed; WLS reduces variance (H6)
[saved] outputs/T5_heteroscedasticity.csv


[figure] outputs/figures/T5_heteroscedasticity.png
T5 in 387.6s


---
## 06 - T6: Chebyshev vs c-optimal design (Hypothesis H5, PO4)

**Hypothesis:** the Chebyshev nodes answer a **different question**. They minimize the
sup-norm of the *interpolation* error of a known function, with uniform weight. The
object here is the **variance of the derivative functional at a point**, under
heteroscedastic noise. The correct criterion is the **c-optimal design** (Elfving 1952;
Pukelsheim 2006; Dette & Holland-Letz 2009 for the heteroscedastic case), whose weights
are exactly the shot allocation.

This reframes deliverable E4 of the issue: it is not about "reconciling the exploration
points with Chebyshev", but about **replacing Chebyshev** with the criterion that
corresponds to the actual target.

**Pass criterion:** `Var_c-optimal / Var_Chebyshev < 0.8`.

In [12]:
# --- T6: c-optimal design ---
import numpy as np, pandas as pd, math
from scipy.optimize import minimize
def design_basis(x, deg, x_eval, scale):
    """
    Polynomial basis CENTERED at x_eval and SCALED by `scale`.

    Without this the design matrix is severely ill-conditioned: p ~ 1e-2 raised to
    powers produces columns with scales separated by orders of magnitude, and the
    c-optimal design degenerates. With z = (x - x_eval)/scale, the derivative at x_eval
    is simply coef[1]/scale, so c = e_1 / scale.
    """
    z = (np.asarray(x, float) - x_eval) / scale
    F = np.vstack([z ** j for j in range(deg + 1)]).T
    c = np.zeros(deg + 1); c[1] = 1.0 / scale
    return F, c


def c_optimal_design(grid, sigma, deg, x_eval):
    """Minimize c' M(w)^-1 c with M = sum_k w_k f_k f_k' / sigma_k^2. Returns weights."""
    scale = max((float(np.max(grid)) - float(np.min(grid))) / 2.0, 1e-12)
    F, c = design_basis(grid, deg, x_eval, scale)
    inv_s2 = 1.0 / np.asarray(sigma, float) ** 2

    def obj(w):
        M = (F * (np.abs(w) * inv_s2)[:, None]).T @ F
        try:
            return float(c @ np.linalg.solve(M + 1e-13 * np.eye(deg + 1), c))
        except np.linalg.LinAlgError:
            return 1e12

    n = len(grid)
    res = minimize(obj, np.ones(n) / n, method="SLSQP", bounds=[(0.0, 1.0)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
                   options={"maxiter": 400, "ftol": 1e-12})
    w = np.clip(res.x, 0, None)
    w = w / w.sum()
    return w, obj(w)


def chebyshev_nodes(a, b, n):
    k = np.arange(1, n + 1)
    return np.sort((a + b) / 2 + (b - a) / 2 * np.cos((2 * k - 1) / (2 * n) * np.pi))


def design_variance(grid, w, sigma, deg, x_eval):
    scale = max((float(np.max(grid)) - float(np.min(grid))) / 2.0, 1e-12)
    F, c = design_basis(grid, deg, x_eval, scale)
    M = (F * (w / np.asarray(sigma, float) ** 2)[:, None]).T @ F
    return float(c @ np.linalg.solve(M + 1e-13 * np.eye(deg + 1), c))


def weighted_derivative_fit(xs, ys, ws, deg, x_eval, scale=None):
    xs = np.asarray(xs, float)
    scale = scale or max((xs.max() - xs.min()) / 2.0, 1e-12)
    F, c = design_basis(xs, deg, x_eval, scale)
    W = np.diag(np.asarray(ws, float))
    coef = np.linalg.solve(F.T @ W @ F + 1e-12 * np.eye(deg + 1), F.T @ W @ np.asarray(ys, float))
    return float(coef[1] / scale)


print("c-optimal design defined")

c-optimal design defined


In [13]:
# --- T6: comparison of the three designs ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd
t0 = time.time()
DEG = 2
grid_fine = np.linspace(A_T5, B_T5, 25)
sigma_fine = np.interp(grid_fine, grid_t5, sig_g)

w_copt, v_copt = c_optimal_design(grid_fine, sigma_fine, DEG, P_STAR[IDX_BULK])

cheb = chebyshev_nodes(A_T5, B_T5, DEG + 1)
w_cheb = np.zeros(len(grid_fine))
for x in cheb:
    w_cheb[int(np.argmin(abs(grid_fine - x)))] += 1.0 / (DEG + 1)
v_cheb = design_variance(grid_fine, w_cheb, sigma_fine, DEG, P_STAR[IDX_BULK])

expo = P_STAR[IDX_BULK] * np.exp(np.array([-1.0, -0.45, -0.2, 0.2, 0.45, 1.0]))
expo = expo[(expo >= A_T5) & (expo <= B_T5)]
w_expo = np.zeros(len(grid_fine))
for x in expo:
    w_expo[int(np.argmin(abs(grid_fine - x)))] += 1.0 / len(expo)
v_expo = design_variance(grid_fine, w_expo, sigma_fine, DEG, P_STAR[IDX_BULK])

print("Var(derivative) by design (smaller = better):")
print(f"  c-optimal               = {v_copt:.5g}")
print(f"  Chebyshev               = {v_cheb:.5g}   (ratio c-opt/cheb = {v_copt/v_cheb:.3f})")
print(f"  exponential reuse       = {v_expo:.5g}   (ratio c-opt/expo = {v_copt/v_expo:.3f})")
print(f"\nc-optimal support: {np.round(grid_fine[w_copt > 1e-3], 5)}")
print(f"weights             : {np.round(w_copt[w_copt > 1e-3], 3)}")

ok_t6 = verdict(v_copt / v_cheb < 0.8,
                f"c-optimal beats Chebyshev by {v_cheb/v_copt:.2f}x (H5/PO4 confirmed)",
                "c-optimal did not beat Chebyshev - H5 weakened")

df_t6 = pd.DataFrame(dict(design=["c_optimal", "chebyshev", "exponential_reuse"],
                          variance=[v_copt, v_cheb, v_expo],
                          ratio_vs_chebyshev=[v_copt/v_cheb, 1.0, v_expo/v_cheb]))
save_table(df_t6, "T6_design")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df_t6.design, df_t6.variance, color=["#2a9d8f", "#e76f51", "#e9c46a"])
for i, v in enumerate(df_t6.variance):
    ax.text(i, v, f"{v:.3g}", ha="center", va="bottom")
ax.set_ylabel("Var(derivative)"); ax.set_title("Design variance comparison (c-optimal vs Chebyshev vs exponential)")
save_fig(fig, "T6_design_variance")
RESULTS["T6"] = dict(var_copt=v_copt, var_cheb=v_cheb, var_expo=v_expo,
                     gain=v_cheb / v_copt, passed=ok_t6)
print(f"T6 in {time.time() - t0:.1f}s")

Var(derivative) by design (smaller = better):
  c-optimal               = 1099.3
  Chebyshev               = 4767.2   (ratio c-opt/cheb = 0.231)
  exponential reuse       = 3139   (ratio c-opt/expo = 0.350)

c-optimal support: [0.01162 0.02344 0.0285 ]
weights             : [0.733 0.218 0.048]
  >> VALIDATED: c-optimal beats Chebyshev by 4.34x (H5/PO4 confirmed)
[saved] outputs/T6_design.csv


[figure] outputs/figures/T6_design_variance.png
T6 in 4.9s


---
## 07 - T7: Validity guard of Lambda (Hypothesis H8)

**Hypothesis:** the issue lists **two** conditions for the bounds (Lambda varies enough;
`p_L` in the feasible range). A third one is missing: **`p_L proportional Lambda^{-(d+1)/2}`
is asymptotic and only holds well below threshold.** Widening toward the threshold, the
SNR *improves* precisely where Lambda stops meaning anything. The proposed algorithm is
**attracted** to the region where its answer has no meaning.

**Pass criterion:** there exists a region where the SNR of DeltaLambda grows at the same
time as the R2 of the `log p_L x d` fit degrades below 0.95.

In [14]:
# --- T7: SNR rises while the validity of Lambda falls ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd
t0 = time.time()
scan = P_STAR[IDX_BULK] * np.array([1.0, 1.5, 2.2, 3.0, 4.0, 5.5, 7.0, 9.0])
rows_t7 = []
for g in scan:
    pv = P_STAR.copy(); pv[IDX_BULK] = g
    lam_i, sig_i, r2_i, pLs_i = TB.lam(pv, SHOTS, GLOBAL_SEED + 71)
    L0, _, _, _ = TB.lam(P_STAR, SHOTS, GLOBAL_SEED + 71)
    snr_i = abs(lam_i - L0) / max(sig_i, 1e-9)
    rows_t7.append(dict(p_bulk=g, ratio_to_p_star=g / P_STAR[IDX_BULK],
                        lambda_hat=lam_i, sigma=sig_i, r2_lambda_fit=r2_i,
                        snr_vs_p_star=snr_i, pL_max=max(pLs_i)))
    print(f"  p={g:.4f} ({g/P_STAR[IDX_BULK]:4.1f}x): Lambda={lam_i:6.3f} "
          f"R2={r2_i:.4f}  SNR={snr_i:6.1f}  pL_max={max(pLs_i):.4f}")

df_t7 = pd.DataFrame(rows_t7)
bad = df_t7[df_t7.r2_lambda_fit < 0.95]
ok_t7 = verdict(len(bad) > 0 and bool((bad.snr_vs_p_star > 3).any()),
                "there is a region with high SNR and invalid Lambda model (H8 confirmed)",
                "could not reach the invalidity regime in this scan")
save_table(df_t7, "T7_lambda_validity")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_t7.ratio_to_p_star, df_t7.snr_vs_p_star, "o-", color="#e76f51", label="SNR vs p*")
ax.axhline(3, color="r", ls="--", lw=1)
ax2 = ax.twinx()
ax2.plot(df_t7.ratio_to_p_star, df_t7.r2_lambda_fit, "s--", color="#2a9d8f", label="R2 of log pL fit")
ax2.axhline(0.95, color="g", ls="--", lw=1)
ax.set_xlabel("p_bulk / p*"); ax.set_ylabel("SNR", color="#e76f51")
ax2.set_ylabel("R2", color="#2a9d8f")
ax.set_title("SNR rises while Lambda model validity falls (H8)")
lines1, lab1 = ax.get_legend_handles_labels(); lines2, lab2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, lab1 + lab2, loc="best")
save_fig(fig, "T7_snr_vs_validity")
RESULTS["T7"] = dict(n_invalid_points=int(len(bad)), passed=ok_t7,
                     r2_min=float(df_t7.r2_lambda_fit.min()))
print(f"T7 in {time.time() - t0:.1f}s")

  p=0.0150 ( 1.0x): Lambda= 3.319 R2=0.9999  SNR=   0.0  pL_max=0.0154


  p=0.0225 ( 1.5x): Lambda= 1.978 R2=1.0000  SNR=  28.0  pL_max=0.0227


  p=0.0330 ( 2.2x): Lambda= 1.312 R2=0.9929  SNR=  98.0  pL_max=0.0341


  p=0.0450 ( 3.0x): Lambda= 0.952 R2=0.8703  SNR= 222.6  pL_max=0.0526


  p=0.0600 ( 4.0x): Lambda= 0.764 R2=0.9664  SNR= 388.7  pL_max=0.1116


  p=0.0825 ( 5.5x): Lambda= 0.647 R2=0.9713  SNR= 608.2  pL_max=0.2149


  p=0.1050 ( 7.0x): Lambda= 0.629 R2=0.9605  SNR= 721.2  pL_max=0.2901


  p=0.1350 ( 9.0x): Lambda= 0.641 R2=0.9437  SNR= 777.4  pL_max=0.3453
  >> VALIDATED: there is a region with high SNR and invalid Lambda model (H8 confirmed)
[saved] outputs/T7_lambda_validity.csv


[figure] outputs/figures/T7_snr_vs_validity.png
T7 in 39.7s


---
## 08 - T8: Lambda as a second-level estimator (Hidden Problem PO2)

**Hypothesis:** Lambda is **not an observable** - it is the result of a fit
(`log p_L x d`). The budget uncertainty is propagation through **two nested fits**. If
the `sigma_Lambda` reported by the fit underestimates the empirical `sigma_Lambda`, the
budget error bars are falsely narrow.

**Pass criterion:** `sigma_bootstrap / sigma_delta-method > 1.2`.

In [15]:
# --- T8: bootstrap of Lambda vs delta-method ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd
t0 = time.time()
lam_boot = []
for b in range(N_BOOT):
    l, _, _, _ = TB.lam(P_STAR, SHOTS, GLOBAL_SEED + 5000 + b)
    lam_boot.append(l)
lam_boot = np.array(lam_boot)

_, sig_delta, _, _ = TB.lam(P_STAR, SHOTS, GLOBAL_SEED)
sig_boot = float(lam_boot.std(ddof=1))
ratio_t8 = sig_boot / sig_delta if sig_delta > 0 else float("nan")

print(f"mean Lambda (bootstrap) = {lam_boot.mean():.4f}")
print(f"empirical sigma (bootstrap, N={N_BOOT}) = {sig_boot:.5f}")
print(f"reported sigma (delta-method)          = {sig_delta:.5f}")
print(f"\nRatio bootstrap/delta = {ratio_t8:.3f}")
ok_t8 = verdict(ratio_t8 > 1.2,
                "delta-method underestimates sigma_Lambda (PO2 confirmed)",
                "delta-method is consistent with the bootstrap in this regime")

df_t8 = pd.DataFrame(dict(replica=range(N_BOOT), lambda_hat=lam_boot))
save_table(df_t8, "T8_bootstrap_lambda")

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(lam_boot, kde=True, color="#2a9d8f", ax=ax)
ax.axvline(lam_boot.mean(), color="k", ls="-", label="bootstrap mean")
ax.axvspan(lam_boot.mean()-sig_boot, lam_boot.mean()+sig_boot, alpha=0.15, color="#e76f51", label="bootstrap sigma")
ax.set_xlabel("Lambda"); ax.set_title(f"Bootstrap Lambda (sigma_boot/sigma_delta = {ratio_t8:.3f})")
ax.legend()
save_fig(fig, "T8_bootstrap")
RESULTS["T8"] = dict(sigma_bootstrap=sig_boot, sigma_delta=sig_delta,
                     ratio=ratio_t8, passed=ok_t8)
print(f"T8 in {time.time() - t0:.1f}s")

mean Lambda (bootstrap) = 3.2262
empirical sigma (bootstrap, N=400) = 0.16604
reported sigma (delta-method)          = 0.12123

Ratio bootstrap/delta = 1.370
  >> VALIDATED: delta-method underestimates sigma_Lambda (PO2 confirmed)
[saved] outputs/T8_bootstrap_lambda.csv


[figure] outputs/figures/T8_bootstrap.png
T8 in 639.9s


---
## 09 - DELIVERABLE E1 + E2 + E3: `find_bounds_auto`

This is the prototype of what the issue asks. It implements deliverables 1, 2 and 3 and
incorporates the corrections from the previous tests:

| Correction | Origin |
|-----------|--------|
| **Log-space** loop (automatic positivity) | T4 / H7 |
| `sigma_DeltaLambda` measured under **CRN**, not inferred from `sqrt(2)*sigma_Lambda` | T1 / H2 |
| `p_L` guard at both ends (feasible range) | issue, condition 2 |
| **Validity guard of the Lambda model** (R2 of the fit) | T7 / H8 |
| Explicit verdict of **insensitive parameter** | E5 |
| Full diagnostic returned (nothing silent) | H10 / PO6 |

The stop criterion is dual: the signal must exceed the **coupled** noise (`SNR`) **and**
an absolute floor relative to `sigma_Lambda`. SNR alone is not enough - for a parameter
that affects nothing, CRN gives `DeltaLambda = 0` with deviation `0`, and a pure-ratio
criterion would return `0/0`.

In [16]:
# --- E1+E2+E3: find_bounds_auto ---
def find_bounds_auto(tb, pvec, i, shots=None, eps0=0.05, gamma=1.6,
                     snr_target=3.0, signal_floor=0.5,
                     pL_min=2e-4, pL_max=0.40, r2_min=0.90, r2_drop=0.05,
                     eps_max=1.6, n_rep=3, max_iter=12, seed=0):
    """
    Adaptive log-space bracketing for parameter i.

    Returns (a, b), diagnostics.

    Stop criteria, in precedence order:
      guard_pL_too_small / guard_pL_too_large  -> Monte Carlo range (issue, cond. 2)
      guard_lambda_model_invalid               -> R2 of log pL x d fit dropped (H8)
      snr_satisfied                            -> signal exceeds coupled noise (issue, cond. 1)
      eps_cap + insensitive                    -> parameter does not move Lambda (E5)
    """
    shots = shots or SHOTS
    p_i = float(pvec[i])
    eps = float(eps0)
    _, _, r2_ref, _ = tb.lam(pvec, shots, int(seed * 1000 + 999))
    r2_gate = min(r2_min, r2_ref - r2_drop)
    diag = dict(param_index=i, param_name=tb.param_names[i], iters=0,
                stop_reason="max_iter", history=[], shots_used=0,
                snr=float("nan"), insensitive=False, binding_constraint=None,
                r2_reference=None, r2_gate=None)
    best = (p_i * math.exp(-eps), p_i * math.exp(eps))
    diag["r2_reference"] = r2_ref
    diag["r2_gate"] = r2_gate

    for it in range(max_iter):
        a, b = p_i * math.exp(-eps), p_i * math.exp(eps)
        pa, pb = pvec.copy(), pvec.copy()
        pa[i], pb[i] = a, b

        diffs, r2s, pL_all, sig_all = [], [], [], []
        for r in range(n_rep):
            s = int(seed * 1000 + it * 50 + r)
            La, sa, r2a, pLa = tb.lam(pa, shots, s)
            Lb, sb, r2b, pLb = tb.lam(pb, shots, s)      # CRN: same seed
            diffs.append(Lb - La)
            r2s += [r2a, r2b]; pL_all += list(pLa) + list(pLb); sig_all += [sa, sb]
        diag["shots_used"] += 2 * n_rep * shots * len(tb.distances)

        dL = float(np.mean(diffs))
        sd = float(np.std(diffs, ddof=1)) if n_rep > 1 else float("inf")
        snr = abs(dL) / sd if sd > 1e-12 else (0.0 if abs(dL) < 1e-12 else float("inf"))
        sig_lam = float(np.mean(sig_all))
        pL_lo, pL_hi, r2_lo = float(min(pL_all)), float(max(pL_all)), float(min(r2s))

        diag["history"].append(dict(iter=it, eps=eps, a=a, b=b, dLambda=dL, sd_crn=sd,
                                    snr=snr, sigma_lambda=sig_lam, pL_min=pL_lo,
                                    pL_max=pL_hi, r2_min=r2_lo))
        diag["iters"] = it + 1
        diag["snr"] = snr

        if pL_lo < pL_min:
            diag["stop_reason"] = "guard_pL_too_small"; diag["binding_constraint"] = "pL_min"; break
        if pL_hi > pL_max:
            diag["stop_reason"] = "guard_pL_too_large"; diag["binding_constraint"] = "pL_max"; break
        if r2_lo < r2_gate:
            diag["stop_reason"] = "guard_lambda_model_invalid"; diag["binding_constraint"] = "r2"; break

        best = (a, b)
        if snr >= snr_target and abs(dL) >= signal_floor * sig_lam:
            diag["stop_reason"] = "snr_satisfied"; diag["binding_constraint"] = "snr"; break
        if eps >= eps_max:
            diag["stop_reason"] = "eps_cap"; diag["binding_constraint"] = "eps_max"
            diag["insensitive"] = abs(dL) < signal_floor * sig_lam
            break
        eps = min(eps * gamma, eps_max)

    diag["bounds"] = best
    return best, diag


def get_error_budget_gradient(tb, pvec, i, bounds=None, deg=2, shots=None,
                              design="c_optimal", seed=0):
    """
    E2: signature with OPTIONAL `bounds`.
    If bounds is None, triggers automatic exploration. Always returns diagnostics.
    """
    shots = shots or SHOTS
    diag_bounds = None
    if bounds is None:
        bounds, diag_bounds = find_bounds_auto(tb, pvec, i, shots=shots, seed=seed)
    a, b = bounds
    grid = np.linspace(a, b, 21)
    sig = []
    for g in grid[::4]:
        pv = pvec.copy(); pv[i] = g
        _, s, _, _ = tb.lam(pv, shots, seed + 1)
        sig.append(s)
    sig_fine = np.interp(grid, grid[::4], sig)

    if design == "c_optimal":
        w, _ = c_optimal_design(grid, sig_fine, deg, pvec[i])
    else:
        w = np.zeros(len(grid))
        for x in chebyshev_nodes(a, b, deg + 1):
            w[int(np.argmin(abs(grid - x)))] += 1.0 / (deg + 1)

    sup = np.where(w > 1e-3)[0]
    xs, ys, ws = [], [], []
    for k in sup:
        pv = pvec.copy(); pv[i] = grid[k]
        n_k = max(int(shots * w[k] * len(sup)), 2_000)
        l, s, _, _ = tb.lam(pv, n_k, seed + 2)
        xs.append(grid[k]); ys.append(l); ws.append(1.0 / max(s, 1e-9) ** 2)

    grad = weighted_derivative_fit(xs, ys, ws, deg, pvec[i], scale=(b - a) / 2)

    if diag_bounds is not None and diag_bounds.get("insensitive"):
        return dict(gradient=0.0, insensitive=True, bounds=(a, b), design=design,
                    degree=deg, design_support=[], design_weights=[],
                    bounds_auto=True, bounds_diagnostics=diag_bounds)

    return dict(gradient=grad, insensitive=False, bounds=(a, b), design=design, degree=deg,
                design_support=list(xs), design_weights=list(w[sup]),
                bounds_auto=bounds is not None and diag_bounds is not None,
                bounds_diagnostics=diag_bounds)


print("find_bounds_auto / get_error_budget_gradient defined")

find_bounds_auto / get_error_budget_gradient defined


In [17]:
# --- E1+E3: run autobound over all parameters ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd, time, math
t0 = time.time()
rows_e1 = []
bounds_found = {}
for i, nm in enumerate(TB.param_names):
    ta = time.time()
    (a, b), dg = find_bounds_auto(TB, P_STAR, i, shots=SHOTS, seed=GLOBAL_SEED % 97 + i)
    bounds_found[nm] = (a, b)
    rows_e1.append(dict(param=nm, bound_lo=a, bound_hi=b,
                        width_rel=(b - a) / P_STAR[i], iters=dg["iters"],
                        stop_reason=dg["stop_reason"], binding=dg["binding_constraint"],
                        snr=dg["snr"], insensitive=dg["insensitive"],
                        shots_used=dg["shots_used"], seconds=time.time() - ta))
    flag = "  [INSENSITIVE]" if dg["insensitive"] else ""
    print(f"{nm:>9}: ({a:.5f}, {b:.5f})  width={100*(b-a)/P_STAR[i]:5.1f}%  "
          f"iters={dg['iters']}  stop={dg['stop_reason']}{flag}")

df_e1 = pd.DataFrame(rows_e1)
save_table(df_e1, "E1_autobound")

hist = []
for i, nm in enumerate(TB.param_names):
    _, dg = find_bounds_auto(TB, P_STAR, i, shots=SHOTS // 2, seed=GLOBAL_SEED % 89 + i)
    for h in dg["history"]:
        h["param"] = nm; hist.append(h)
save_table(pd.DataFrame(hist), "E1_autobound_history")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df_e1.param, df_e1.width_rel, color=["#2a9d8f", "#e76f51", "#e9c46a"])
for i, v in enumerate(df_e1.width_rel):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")
ax.set_ylabel("Bracket relative width (b-a)/p"); ax.set_title("Automatic bracket width per parameter")
save_fig(fig, "E1_autobound_widths")

RESULTS["E1"] = dict(bounds=bounds_found,
                     stop_reasons={r["param"]: r["stop_reason"] for r in rows_e1},
                     insensitive_detected=bool(df_e1.insensitive.any()))
print(f"\nE1 in {time.time() - t0:.1f}s")

 boundary: (0.00888, 0.02534)  width=109.7%  iters=6  stop=snr_satisfied


     bulk: (0.01427, 0.01577)  width= 10.0%  iters=1  stop=snr_satisfied


   dummy0: (0.00303, 0.07430)  width=475.1%  iters=9  stop=eps_cap  [INSENSITIVE]
[saved] outputs/E1_autobound.csv


[saved] outputs/E1_autobound_history.csv


[figure] outputs/figures/E1_autobound_widths.png

E1 in 236.0s


---
## 10 - DELIVERABLE E4: reconciliation with the fitting

The issue marks this point as explicitly non-trivial: *"these points sit far away from
the Chebyshev nodes, which are ideal for fitting"*.

Here the decision is **empirical**, comparing three strategies over the **same bracket**
and the **same shot budget**:

- **A - reuse** the exponential exploration points;
- **B - resample at Chebyshev nodes**;
- **C - resample at weighted c-optimal design** (this analysis's proposal).

**Pass criterion:** C has the lowest empirical variance of the derivative.

In [18]:
# --- E4: benchmark of the three reconciliation strategies ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd, math
t0 = time.time()
a_e4, b_e4 = bounds_found["bulk"]
if (b_e4 - a_e4) / P_STAR[IDX_BULK] < 0.40:
    a_e4 = P_STAR[IDX_BULK] * 0.70
    b_e4 = P_STAR[IDX_BULK] * 1.45
print(f"bracket used: ({a_e4:.5f}, {b_e4:.5f})")

grid_e4 = np.linspace(a_e4, b_e4, 21)
sig_e4 = []
for g in grid_e4[::5]:
    pv = P_STAR.copy(); pv[IDX_BULK] = g
    _, s, _, _ = TB.lam(pv, SHOTS, GLOBAL_SEED + 611)
    sig_e4.append(s)
sig_e4 = np.interp(grid_e4, grid_e4[::5], sig_e4)

designs = {}
w_c, _ = c_optimal_design(grid_e4, sig_e4, 2, P_STAR[IDX_BULK])
designs["C_c_optimal"] = w_c

w_ch = np.zeros(len(grid_e4))
for x in chebyshev_nodes(a_e4, b_e4, 3):
    w_ch[int(np.argmin(abs(grid_e4 - x)))] += 1 / 3
designs["B_chebyshev"] = w_ch

expo_pts = P_STAR[IDX_BULK] * np.exp(np.linspace(math.log(a_e4 / P_STAR[IDX_BULK]),
                                                 math.log(b_e4 / P_STAR[IDX_BULK]), 5))
w_ex = np.zeros(len(grid_e4))
for x in expo_pts:
    w_ex[int(np.argmin(abs(grid_e4 - x)))] += 1 / len(expo_pts)
designs["A_exponential_reuse"] = w_ex

TOTAL_SHOTS_E4 = SHOTS * 6
rows_e4 = []
for name, w in designs.items():
    sup = np.where(w > 1e-3)[0]
    ders = []
    for r in range(N_REP):
        xs, ys, ws = [], [], []
        for k in sup:
            pv = P_STAR.copy(); pv[IDX_BULK] = grid_e4[k]
            n_k = max(int(TOTAL_SHOTS_E4 * w[k]), 1_000)
            l, s, _, _ = TB.lam(pv, n_k, GLOBAL_SEED + 7000 + r)
            xs.append(grid_e4[k]); ys.append(l); ws.append(1.0 / max(s, 1e-9) ** 2)
        try:
            ders.append(weighted_derivative_fit(xs, ys, ws, 2, P_STAR[IDX_BULK],
                                                scale=(b_e4 - a_e4) / 2))
        except np.linalg.LinAlgError:
            ders.append(np.nan)
    ders = np.array([d for d in ders if np.isfinite(d)])
    rows_e4.append(dict(strategy=name, n_support=int(len(sup)),
                        mean_gradient=float(ders.mean()),
                        std_gradient=float(ders.std(ddof=1)),
                        var_gradient=float(ders.var(ddof=1))))
    print(f"{name:>22}: points={len(sup)}  grad={ders.mean():+9.2f} +- {ders.std(ddof=1):8.2f}")

df_e4 = pd.DataFrame(rows_e4).sort_values("var_gradient")
best_e4 = df_e4.iloc[0]["strategy"]
print(f"\nLowest variance: {best_e4}")
ok_e4 = verdict(best_e4 == "C_c_optimal",
                "resampling at c-optimal design wins - the issue tension is resolved by SUBSTITUTION",
                f"the winning strategy was {best_e4}, not the c-optimal design")
save_table(df_e4, "E4_fitting_reconciliation")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df_e4.strategy, df_e4.var_gradient, color=["#2a9d8f", "#e76f51", "#e9c46a"])
for i, v in enumerate(df_e4.var_gradient):
    ax.text(i, v, f"{v:.3g}", ha="center", va="bottom")
ax.set_ylabel("Var(derivative)"); ax.set_title("Reconciliation benchmark: variance of derivative by strategy")
save_fig(fig, "E4_reconciliation")
RESULTS["E4"] = dict(winner=best_e4, rows=rows_e4, passed=ok_e4)
print(f"E4 in {time.time() - t0:.1f}s")

bracket used: (0.01050, 0.02175)


           C_c_optimal: points=2  grad=  -289.12 +-    18.86


           B_chebyshev: points=3  grad=  -289.55 +-    34.10


   A_exponential_reuse: points=5  grad=  -282.91 +-    30.48

Lowest variance: C_c_optimal
  >> VALIDATED: resampling at c-optimal design wins - the issue tension is resolved by SUBSTITUTION
[saved] outputs/E4_fitting_reconciliation.csv


[figure] outputs/figures/E4_reconciliation.png
E4 in 869.8s


---
## 11 - DELIVERABLE E5: boundary regimes

The issue asks for tests covering an **insensitive parameter** (the algorithm must
degrade gracefully, not widen forever) and a **very sensitive parameter**.

The three regimes are tested on the same testbed:

| Regime | Parameter | Expected behavior |
|--------|-----------|-------------------|
| Insensitive (exact) | `dummy0` - no mechanism | stops at `eps_cap`, marks `insensitive=True`, returns no spurious gradient |
| Weakly sensitive | `boundary` | widens several iterations, converges |
| Highly sensitive | `bulk` | stops in 1-2 iterations with a narrow bracket |

**Reproducibility** (H10) is also tested: the same seed must give the same bracket.

In [19]:
# --- E5: boundary regimes + reproducibility ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd, time
t0 = time.time()
rows_e5 = []
for i, nm in enumerate(TB.param_names):
    (a, b), dg = find_bounds_auto(TB, P_STAR, i, shots=SHOTS, seed=11 + i)
    res = get_error_budget_gradient(TB, P_STAR, i, bounds=None, shots=SHOTS, seed=11 + i)
    expected = {"dummy0": "insensitive", "boundary": "weakly_sensitive", "bulk": "highly_sensitive"}[nm]
    rows_e5.append(dict(param=nm, regime=expected, bound_lo=a, bound_hi=b,
                        iters=dg["iters"], stop_reason=dg["stop_reason"],
                        insensitive_flag=dg["insensitive"] or res.get("insensitive", False),
                        gradient=res["gradient"]))
    print(f"{nm:>9} [{expected:>15}]: iters={dg['iters']:2d} stop={dg['stop_reason']:>26} "
          f"insens={str(dg['insensitive']):>5} grad={res['gradient']:+10.2f}")

df_e5 = pd.DataFrame(rows_e5)

(bnds_a, _) = find_bounds_auto(TB, P_STAR, IDX_BULK, shots=SHOTS, seed=4242)
(bnds_b, _) = find_bounds_auto(TB, P_STAR, IDX_BULK, shots=SHOTS, seed=4242)
(bnds_c, _) = find_bounds_auto(TB, P_STAR, IDX_BULK, shots=SHOTS, seed=9999)
repro = (bnds_a == bnds_b)
print(f"\nReproducibility: same seed -> same bounds? {repro}")
print(f"  seed=4242: {tuple(round(v,6) for v in bnds_a)}")
print(f"  seed=9999: {tuple(round(v,6) for v in bnds_c)}")

dummy_row = df_e5[df_e5.param == "dummy0"].iloc[0]
ok_e5 = verdict(bool(dummy_row.insensitive_flag) and repro,
                "insensitive parameter explicitly detected and run reproducible (E5+H10)",
                "insensitive regime not flagged or run not reproducible")
save_table(df_e5, "E5_boundary_regimes")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df_e5.param, df_e5.iters, color=["#e9c46a", "#e76f51", "#2a9d8f"])
for i, v in enumerate(df_e5.iters):
    ax.text(i, v, f"{v}", ha="center", va="bottom")
ax.set_ylabel("Iterations"); ax.set_title("Boundary regimes: iterations to stop per parameter")
save_fig(fig, "E5_regimes")
RESULTS["E5"] = dict(rows=rows_e5, reproducible=bool(repro), passed=ok_e5)
print(f"E5 in {time.time() - t0:.1f}s")

 boundary [weakly_sensitive]: iters= 1 stop=             snr_satisfied insens=False grad=   +152.01


     bulk [highly_sensitive]: iters= 1 stop=             snr_satisfied insens=False grad=   -325.51


   dummy0 [    insensitive]: iters= 9 stop=                   eps_cap insens= True grad=     +0.00



Reproducibility: same seed -> same bounds? True
  seed=4242: (0.014268, 0.015769)
  seed=9999: (0.014268, 0.015769)
  >> VALIDATED: insensitive parameter explicitly detected and run reproducible (E5+H10)
[saved] outputs/E5_boundary_regimes.csv


[figure] outputs/figures/E5_regimes.png
E5 in 311.1s


---
## 12 - Integration with the real Deltakit API (Deliverable E2)

This section leaves the testbed and touches the public API the issue wants to modify:

```python
get_error_budget(noise_model, P, num_rounds_per_distance, bounds, sampling_parameters=...)
```

The goals: (a) reproduce the documented baseline, (b) demonstrate the signature with
**optional** `bounds`, and (c) show that the current return does not expose the
diagnostics that autobound must return (PO6).

This cell is **fault-tolerant**: if `pip install deltakit` fails (e.g., no wheel for the
active Python), the suite continues and the final report records the section as skipped.

In [20]:
# --- Deltakit: installation (may fail if unavailable) ---
import subprocess, sys
print("Attempting to install deltakit...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "deltakit"])
    print("deltakit installed")
except Exception as exc:
    print(f"deltakit install failed: {type(exc).__name__}: {exc}")

DELTAKIT_OK = False
try:
    from deltakit_circuit import Circuit, NoiseProfile
    from deltakit_circuit.noise_channels import Depolarise1, Depolarise2
    from deltakit_explorer.qpu import NoiseParameters, QPU
    from deltakit.explorer.analysis.error_budget import get_error_budget, SamplingParameters
    from deltakit_explorer.analysis.error_budget import inverse_lambda_at
    DELTAKIT_OK = True
    print("deltakit imported successfully")
except Exception as exc:
    print(f"deltakit unavailable: {type(exc).__name__}: {exc}")
    print("Section 12 will be skipped. Sections 00-11 do not depend on deltakit.")

Attempting to install deltakit...


deltakit installed


deltakit unavailable: ModuleNotFoundError: No module named 'deltakit.explorer.analysis.error_budget'
Section 12 will be skipped. Sections 00-11 do not depend on deltakit.


In [21]:
# --- Deltakit: baseline + optional bounds ---
import numpy as np, pandas as pd, math, time
if DELTAKIT_OK:
    import numpy.typing as npt

    def simple_noise_model(circuit, noise_parameters):
        gate_noise = [
            lambda nc: Depolarise1.generator_from_prob(noise_parameters[0])(
                nc.gate_layer_qubits(None, gate_qubit_count=1)),
            lambda nc: Depolarise2.generator_from_prob(noise_parameters[0])(
                nc.gate_layer_qubits(None, gate_qubit_count=2)),
        ]
        qpu = QPU(circuit.qubits, noise_model=NoiseParameters(gate_noise=gate_noise))
        return qpu.compile_and_add_noise_to_circuit(circuit)

    P_DK = [5e-3]
    NUM_ROUNDS_DK = {d: [16] for d in [3, 5, 7]}
    BOUNDS_DOC = [(1e-3, 2e-2)]
    MAX_SHOTS_DK = 200_000 if QUICK else 500_000

    try:
        t0 = time.time()
        res_doc = get_error_budget(
            simple_noise_model, P_DK, NUM_ROUNDS_DK, BOUNDS_DOC,
            sampling_parameters=SamplingParameters(max_shots=MAX_SHOTS_DK))
        t_doc = time.time() - t0
        c_doc = float(res_doc.contributions[0])
        s_doc = float(res_doc.contribution_stddevs[0])
        print(f"documented baseline: contribution = {c_doc:.5f} +- {s_doc:.5f}  ({t_doc:.1f}s)")

        def get_error_budget_optional_bounds(noise_model, P, num_rounds, bounds=None,
                                             sampling_parameters=None, **kw):
            """Wrapper demonstrating the requested signature (optional bounds)."""
            auto = None
            if bounds is None:
                import stim as _stim, pymatching as _pm

                def probe_pL(p, d, rounds, shots=40_000):
                    c = _stim.Circuit.generated("surface_code:rotated_memory_z",
                                                rounds=rounds, distance=d,
                                                after_clifford_depolarization=p)
                    dem = c.detector_error_model(decompose_errors=True)
                    m = _pm.Matching(dem)
                    det, obs = c.compile_detector_sampler().sample(
                        shots, separate_observables=True, bit_packed=False)
                    pred = m.decode_batch(det)
                    return float((pred != obs).any(axis=1).mean())

                half_p = P[0] / 2.0
                eps = 0.2
                lo, hi = half_p, half_p
                for _ in range(12):
                    lo, hi = half_p * math.exp(-eps), half_p * math.exp(eps)
                    pl_lo = probe_pL(lo, max(num_rounds), num_rounds[max(num_rounds)][0])
                    pl_hi = probe_pL(hi, min(num_rounds), num_rounds[min(num_rounds)][0])
                    if pl_lo < 2e-4 or pl_hi > 0.40:
                        break
                    if pl_hi / max(pl_lo, 1e-9) > 20:
                        break
                    eps *= 1.5
                lo = min(lo, half_p * 0.5)
                hi = max(hi, P[0] * 2.0)
                bounds = [(lo, hi)]
                auto = dict(explored=True, bounds=bounds[0])
            r = get_error_budget(noise_model, P, num_rounds, bounds,
                                 sampling_parameters=sampling_parameters, **kw)
            return r, bounds, auto

        t0 = time.time()
        res_auto, bnds_auto, auto_diag = get_error_budget_optional_bounds(
            simple_noise_model, P_DK, NUM_ROUNDS_DK, bounds=None,
            sampling_parameters=SamplingParameters(max_shots=MAX_SHOTS_DK))
        t_auto = time.time() - t0
        c_auto = float(res_auto.contributions[0])
        s_auto = float(res_auto.contribution_stddevs[0])
        diff_sigma = abs(c_auto - c_doc) / math.sqrt(s_auto ** 2 + s_doc ** 2)

        print(f"automatic bounds  : {tuple(round(v, 6) for v in bnds_auto[0])}")
        print(f"documented bounds : (0.001, 0.02)")
        print(f"auto contribution : {c_auto:.5f} +- {s_auto:.5f}  ({t_auto:.1f}s)")
        print(f"\nDifference: {diff_sigma:.2f} sigma "
              f"{'-> AGREE' if diff_sigma < 3 else '-> DIVERGE'}")

        attrs = [a for a in dir(res_doc) if not a.startswith("_")]
        print(f"\nAttributes of get_error_budget return: {attrs}")
        has_diag = any(k in attrs for k in ("bounds", "diagnostics", "shots_used", "snr"))
        print(f"Exposes effective bounds/diagnostics? {has_diag}  "
              f"(PO6: {'ok' if has_diag else 'gap confirmed'})")

        df_dk = pd.DataFrame([
            dict(method="manual_documented", bound_lo=1e-3, bound_hi=2e-2,
                 contribution=c_doc, stddev=s_doc, seconds=t_doc),
            dict(method="auto_explored", bound_lo=bnds_auto[0][0], bound_hi=bnds_auto[0][1],
                 contribution=c_auto, stddev=s_auto, seconds=t_auto),
        ])
        save_table(df_dk, "DK_api_integration")
        RESULTS["deltakit"] = dict(ok=True, diff_sigma=float(diff_sigma),
                                   auto_bounds=list(bnds_auto[0]),
                                   returns_diagnostics=bool(has_diag))
    except Exception as exc:
        print(f"deltakit section failed at runtime: {type(exc).__name__}: {exc}")
        RESULTS["deltakit"] = dict(ok=False, error=f"{type(exc).__name__}: {exc}")
else:
    RESULTS["deltakit"] = dict(ok=False, error="import failed / unavailable")
    print("section 12 skipped")

section 12 skipped


---
## 13 - Consolidated report

Consolidates the verdicts, maps each test to the corresponding deliverable of issue
#216 and writes `REPORT.md` + `results.json` + all figures to `outputs/`.

In [22]:
# --- Consolidated report ---
import matplotlib.pyplot as plt, numpy as np, pandas as pd, json, os
MAP = [
    ("T1", "H2  - Common Random Numbers",              "E1, E3, E5"),
    ("T2", "H4  - Score function",                     "E1, E2, E4, E5"),
    ("T3", "H3  - Reweighting + ESS",                  "E1, E3, E4"),
    ("T4", "H7  - Log parametrization",                "E3, E5"),
    ("T5", "H6  - Heteroscedasticity / WLS",           "E1, E3, E4"),
    ("T6", "H5, PO4 - Chebyshev vs c-optimal",         "E4"),
    ("T7", "H8  - Lambda validity guard",              "E1, E3"),
    ("T8", "PO2 - Lambda as 2nd-level estimator",      "E1"),
    ("E4", "Reconciliation with the fitting",          "E4"),
    ("E5", "Boundary regimes + reproducibility",       "E5"),
]

lines = ["# Deltakit issue #216 - Validation suite report", "",
         f"QUICK={QUICK} | SHOTS={SHOTS} | SHOTS_BIG={SHOTS_BIG} | N_REP={N_REP}", "",
         "## Verdicts", "",
         "| Test | Hypothesis / PO | #216 Deliverable | Verdict |",
         "|------|-----------------|------------------|---------|"]

n_pass = n_tot = 0
for key, desc, ent in MAP:
    r = RESULTS.get(key, {})
    p = r.get("passed")
    if p is None:
        v = "not executed"
    else:
        v = "VALIDATED" if p else "not validated"
        n_tot += 1; n_pass += int(bool(p))
    lines.append(f"| {key} | {desc} | {ent} | {v} |")

lines += ["", "## Key numbers", ""]
if "T1" in RESULTS:
    lines.append(f"- **T1 (CRN):** variance of DeltaLambda drops {RESULTS['T1']['reduction']:.1f}x "
                 f"under coupling (ratio {RESULTS['T1']['ratio']:.3f}).")
if "T2" in RESULTS:
    lines.append(f"- **T2 (score function):** agrees with finite differences using "
                 f"{RESULTS['T2']['shot_ratio']:.0f}x fewer shots.")
if "T3" in RESULTS:
    lines.append(f"- **T3 (reweighting):** {RESULTS['T3']['n_points_covered']} sweep points "
                 f"covered by a single sampling campaign.")
if "T5" in RESULTS:
    lines.append(f"- **T5 (heteroscedasticity):** sigma_Lambda varies "
                 f"{RESULTS['T5']['sigma_spread']:.1f}x within the bracket.")
if "T6" in RESULTS:
    lines.append(f"- **T6 (design):** c-optimal beats Chebyshev by "
                 f"{RESULTS['T6']['gain']:.2f}x in derivative variance.")
if "T7" in RESULTS:
    lines.append(f"- **T7 (validity):** {RESULTS['T7']['n_invalid_points']} points with "
                 f"high SNR and Lambda-model R2 below 0.95.")
if "T8" in RESULTS:
    lines.append(f"- **T8 (2nd level):** sigma bootstrap / sigma delta-method = "
                 f"{RESULTS['T8']['ratio']:.2f}.")
if "E1" in RESULTS:
    lines.append(f"- **E1 (autobound):** bounds and stop criteria per parameter: "
                 f"{RESULTS['E1']['stop_reasons']}.")
if "E4" in RESULTS:
    lines.append(f"- **E4 (reconciliation):** winning strategy = {RESULTS['E4']['winner']}.")
if "E5" in RESULTS:
    lines.append(f"- **E5 (boundary):** reproducible = {RESULTS['E5']['reproducible']}.")

dk = RESULTS.get("deltakit", {})
lines += ["", "## Integration with the Deltakit API", ""]
if dk.get("ok"):
    lines.append(f"- automatic vs documented bounds: difference of {dk['diff_sigma']:.2f} sigma")
    lines.append(f"- return exposes bounds/diagnostics: {dk['returns_diagnostics']}")
else:
    lines.append(f"- section skipped: {dk.get('error', 'unknown')}")

lines += ["", "## Reading for the Community Fund proposal", "",
          "1. **T1 + T2 + T3 change the scope of the issue.** If CRN, score function and",
          "   reweighting work, the central dilemma ('narrow interval drowns in noise')",
          "   is in large part an artifact of independent sampling, and the gradient can be",
          "   obtained without a bracket for Pauli models. An honest proposal states this and",
          "   positions `find_bounds_auto` as the generic path, not the only path.",
          "2. **T6 + E4 resolve the point the issue marks as non-trivial** by substitution,",
          "   not reconciliation: Chebyshev answers a different question.",
          "3. **T7 adds a third constraint** the issue does not list and without which the",
          "   stop criterion is attracted to the region where Lambda stops existing.",
          "4. **T8 + PO6** indicate the API return must carry uncertainty and diagnostics,",
          "   not just a number.", ""]

report = "\n".join(lines)
path_md = f"{OUTPUT_DIR}/REPORT.md"
with open(path_md, "w") as f:
    f.write(report)

with open(f"{OUTPUT_DIR}/results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)

print(report)
print("\n" + "=" * 66)
print(f"VERDICTS: {n_pass}/{n_tot} validated")
print(f"Files in {OUTPUT_DIR}:")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {fn}")
print("Figures in " + FIG_DIR + ":")
for fn in sorted(os.listdir(FIG_DIR)):
    print(f"  {fn}")
print("=" * 66)

# Deltakit issue #216 - Validation suite report

QUICK=False | SHOTS=80000 | SHOTS_BIG=240000 | N_REP=30

## Verdicts

| Test | Hypothesis / PO | #216 Deliverable | Verdict |
|------|-----------------|------------------|---------|
| T1 | H2  - Common Random Numbers | E1, E3, E5 | VALIDATED |
| T2 | H4  - Score function | E1, E2, E4, E5 | VALIDATED |
| T3 | H3  - Reweighting + ESS | E1, E3, E4 | VALIDATED |
| T4 | H7  - Log parametrization | E3, E5 | VALIDATED |
| T5 | H6  - Heteroscedasticity / WLS | E1, E3, E4 | VALIDATED |
| T6 | H5, PO4 - Chebyshev vs c-optimal | E4 | VALIDATED |
| T7 | H8  - Lambda validity guard | E1, E3 | VALIDATED |
| T8 | PO2 - Lambda as 2nd-level estimator | E1 | VALIDATED |
| E4 | Reconciliation with the fitting | E4 | VALIDATED |
| E5 | Boundary regimes + reproducibility | E5 | VALIDATED |

## Key numbers

- **T1 (CRN):** variance of DeltaLambda drops 6.8x under coupling (ratio 0.147).
- **T2 (score function):** agrees with finite differences using 10x fewer